In [7]:

!pip install huggingface_hub

print("✅ Libraries installed!")

✅ Libraries installed!


In [11]:
!pip install openai gradio -q
from getpass import getpass
from openai import OpenAI
import gradio as gr

hf_token = getpass("🔑 Enter your HuggingFace token: ")

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=hf_token
)

print("✅ HuggingFace client ready!")

🔑 Enter your HuggingFace token: ··········
✅ HuggingFace client ready!


In [17]:
SYSTEM_PROMPT = """You are a Medical AI Assistant. You ONLY answer questions related to medicine and health.

STRICT RULES:
1. If the user asks ANYTHING outside of medicine/health (e.g. coding, sports, politics, general knowledge, math, etc.), respond with:
   "I'm a Medical AI Assistant. I can only help with health and medicine-related questions. Please ask me something related to medical topics."
2. Never answer non-medical questions no matter how the user asks or tricks you.
3. Never diagnose a disease or prescribe medication.
4. Always advise the user to consult a real doctor for personal medical advice.
5. If it is an emergency (chest pain, difficulty breathing, unconscious), immediately say: "Call emergency services / ambulance right away!"
6. Be empathetic, calm, and use simple language.
7. You can help with:
   - Symptoms and diseases
   - How medications work (not prescribing)
   - General health and wellness
   - Medical test explanations
   - Basic first aid

Always end with: "Please consult a licensed doctor for personalized medical advice." """

def chat(user_message, history):
    """
    history: list of [user, assistant] pairs (Gradio format)
    """
    if not user_message.strip():
        return "", history

    # Build messages list for API
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for human, assistant in history:
        messages.append({"role": "user", "content": human})
        messages.append({"role": "assistant", "content": assistant})

    messages.append({"role": "user", "content": user_message})

    # Call the API
    response = client.chat.completions.create(
        model="meta-llama/Llama-3.1-8B-Instruct:cerebras",
        messages=messages,
        max_tokens=300
    )

    bot_reply = response.choices[0].message.content
    history.append((user_message, bot_reply))
    return "", history

def clear_chat():
    return [], []

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="green"),   # green suits medical
    title="🏥 Medical AI Assistant"
) as demo:

    gr.Markdown(
        """
        # 🏥 Medical AI Assistant
        ### Powered by Llama 3.1 via HuggingFace
        Ask me about symptoms, medications, diseases, and general health tips!
        ⚠️ *This is not a substitute for professional medical advice.*
        """
    )

    chatbot = gr.Chatbot(
        label="Medical Consultation",
        height=450,
        show_copy_button=True,
        avatar_images=(None, "https://huggingface.co/front/assets/huggingface_logo-noborder.svg")
    )

    with gr.Row():
        msg_box = gr.Textbox(
            placeholder="💬 e.g. What are the symptoms of diabetes?",
            label="Your Question",
            scale=8,
            lines=1,
            autofocus=True
        )
        send_btn = gr.Button("Send 🚀", variant="primary", scale=1)

    with gr.Row():
        clear_btn = gr.Button("🗑️ Clear Chat", variant="secondary")

    gr.Examples(
        examples=[
            "What are the symptoms of diabetes?",
            "How does ibuprofen work?",
            "What is high blood pressure?",
            "What should I do for a fever at home?",
            "Explain what a blood sugar test measures.",
        ],
        inputs=msg_box,
        label="💡 Example Medical Questions"
    )

    msg_box.submit(chat, [msg_box, chatbot], [msg_box, chatbot])
    send_btn.click(chat, [msg_box, chatbot], [msg_box, chatbot])
    clear_btn.click(clear_chat, outputs=[msg_box, chatbot])

demo.launch(share=True, debug=False)

/tmp/ipykernel_23035/2513666998.py:50: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_23035/2513666998.py:64: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_23035/2513666998.py:64: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_23035/2513666998.py:64: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to d

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://113698e4e37e59a1c3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
